# FBBP — Analyse de performance
## Chapitre 2 : Analyse des transitions

**Problématique** : FBBP presse et récupère haut — mais est-ce que ces récupérations se transforment en danger réel ?

**Matchs analysés** : J22 → J26

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Style global ──────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f6',
    'axes.grid': True,
    'grid.color': '#e8e8e5',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'xtick.bottom': False,
    'ytick.left': False,
    'axes.labelcolor': '#555',
    'xtick.color': '#555',
    'ytick.color': '#555',
})

# ── Palette ───────────────────────────────────────────────
BLUE   = '#378ADD'
GRAY   = '#B4B2A9'
GREEN  = '#3B6D11'
RED    = '#A32D2D'
ORANGE = '#BA7517'

RES_COLOR = {'W': GREEN, 'D': ORANGE, 'L': RED}

OUTPUT_DIR = '../outputs/chap2_charts'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Librairies chargées')

In [ ]:
# ── Chargement données ─────────────────────────────────────
df_raw = pd.read_csv('../data/01_matches_summary.csv').sort_values('round').reset_index(drop=True)

def get_fbbp_stats(row):
    if row['home_team'] == 'FBBP':
        return pd.Series({
            'xg_fbbp':      row['home_xg'],
            'xg_adv':       row['away_xg'],
            'tirs_fbbp':    row['home_shots'],
            'tirs_adv':     row['away_shots'],
            'dist_tir':     row['home_avg_shot_distance'],
            'dist_tir_adv': row['away_avg_shot_distance'],
            'ppda_fbbp':    row['home_ppda'],
            'ppda_adv':     row['away_ppda'],
            'lieu':         'Dom',
            'adversaire':   row['away_team'],
            'resultat':     'W' if row['home_goals'] > row['away_goals'] else ('D' if row['home_goals'] == row['away_goals'] else 'L'),
            'score':        f"{row['home_goals']}-{row['away_goals']}",
        })
    else:
        return pd.Series({
            'xg_fbbp':      row['away_xg'],
            'xg_adv':       row['home_xg'],
            'tirs_fbbp':    row['away_shots'],
            'tirs_adv':     row['home_shots'],
            'dist_tir':     row['away_avg_shot_distance'],
            'dist_tir_adv': row['home_avg_shot_distance'],
            'ppda_fbbp':    row['away_ppda'],
            'ppda_adv':     row['home_ppda'],
            'lieu':         'Ext',
            'adversaire':   row['home_team'],
            'resultat':     'W' if row['away_goals'] > row['home_goals'] else ('D' if row['away_goals'] == row['home_goals'] else 'L'),
            'score':        f"{row['away_goals']}-{row['home_goals']}",
        })

df = df_raw.apply(get_fbbp_stats, axis=1)
df['journee'] = 'J' + df_raw['round'].astype(str).values
df = df.sort_values('journee').reset_index(drop=True)
df['xg_diff']     = df['xg_fbbp'] - df['xg_adv']
df['xg_per_shot'] = df['xg_fbbp'] / df['tirs_fbbp']
df['label'] = df['journee'] + '\n' + df['adversaire'] + '\n' + df['lieu'] + ' · ' + df['score']
df['groupe'] = df['resultat'].map({'W': 'W/D', 'D': 'W/D', 'L': 'L'})

# Dataset team
df_raw_team  = pd.read_csv('../data/02_teams_stats.csv')
df_team_fbbp = df_raw_team[df_raw_team['team'] == 'FBBP'].copy().reset_index(drop=True)
df_team_adv  = df_raw_team[df_raw_team['team'] != 'FBBP'].copy().reset_index(drop=True)

# Variables transitions
df['rec_high']          = df_team_fbbp['recoveries_high'].values
df['rec_opp_half']      = df_team_fbbp['recoveries_opp_half'].values
df['poss_to_box']       = df_team_fbbp['possessions_to_box'].values
df['prog_passes']       = df_team_fbbp['passes_progressive'].values
df['avg_poss_sec']      = df_team_fbbp['avg_possession_seconds'].values
df['losses_high']       = df_team_fbbp['losses_high'].values
df['poss_surf_pct']     = (df_team_fbbp['possessions_to_box'] / df_team_fbbp['possessions'] * 100).round(1).values
df['poss_surf_pct_adv'] = (df_team_adv['possessions_to_box']  / df_team_adv['possessions']  * 100).round(1).values

# Ratio clé
df['ratio_rec_to_box'] = (df['poss_to_box'] / df['rec_high']).round(2)

print(df[['journee','adversaire','resultat','rec_high','rec_opp_half','poss_to_box','ratio_rec_to_box','avg_poss_sec']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

x = np.arange(len(df))
colors = [RES_COLOR[r] for r in df['resultat']]
bars = ax.bar(x, df['ratio_rec_to_box'], color=colors,
              width=0.55, edgecolor='white', linewidth=1)

for bar, val in zip(bars, df['ratio_rec_to_box']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=10, fontweight='bold',
            color=bar.get_facecolor())

avg = df['ratio_rec_to_box'].mean()
ax.axhline(avg, color='#888', linewidth=1.2, linestyle='--')
ax.text(4.55, avg + 0.02, f'moy. {avg:.2f}', fontsize=9, color='#888')

ax.set_xticks(x)
ax.set_xticklabels(df['label'], fontsize=9)
ax.set_ylim(0, df['ratio_rec_to_box'].max() * 1.25)
ax.set_ylabel('Possessions → surface / récupérations hautes', fontsize=11)
ax.set_title('Exploitation des récupérations hautes', fontsize=13, pad=12, fontweight='bold')

patches = [mpatches.Patch(color=c, label=l)
           for l, c in [('Victoire', GREEN), ('Nul', ORANGE), ('Défaite', RED)]]
ax.legend(handles=patches, fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch2_ratio_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusion chapitre 2

Sur ces 5 matchs, FBBP récupère haut de manière consistante — c'est une force
identifiée. Le problème est dans ce qui suit : dans les deux défaites, moins
d'une récupération haute sur trois débouche sur une possession atteignant la
surface (0.15 vs Orléans, 0.38 vs Versailles).

<hr style="border: none; border-top: 1.5px solid #ccc; margin: 8px 0;">

La variable la plus corrélée au résultat est la capacité à progresser avec le
ballon — mesurée par le nombre de passes progressives par possession. FBBP
dépasse 1.0 passes progressives par possession uniquement contre Rouen (victoire).
Dans tous les autres matchs il reste en dessous, sans jamais transformer ses
récupérations hautes en véritable danger.

**Le pressing existe. L'exploitation ne suit pas.**

**Chapitre 3 : quelles recommandations concrètes pour améliorer cette exploitation ?**

## Chapitre 3 — Séquences prioritaires pour l'analyse vidéo

Sur la base de l'analyse des données, voici les situations à observer en priorité.

**1. Transitions après récupération haute**
Les matchs vs Orléans et Versailles montrent un ratio récup. haute → surface 
très bas (0.15 et 0.38). Observer les séquences post-récupération haute dans 
ces deux matchs pour identifier ce qui bloque la progression.

**2. Phases avec passes progressives élevées**
Le match vs Rouen (1.048 passes progressives/possession) est le seul où FBBP 
progresse efficacement. Comparer ces séquences avec Orléans (0.685) pour 
identifier les différences de placement et de timing.

**3. Récupérations hautes non exploitées**
Isoler spécifiquement les 13 récupérations hautes vs Orléans et les 16 vs 
Versailles — pourquoi débouchent-elles si peu sur la surface ?